In this code I attempt to perform Levin-Petrin procedure for production estimation 

In [220]:
# Import packages 
using Pkg 
using GLM
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using Random
using Distributions
using Optim
using Plots
using ShiftedArrays  # for lag function

In [221]:
# Step 0: import and browse dataset
dataset = CSV.read("op_ready.csv", DataFrame)

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64
1,1,1997,15.2437,12.2967,13.6469,0.0199931,27.4075
2,1,1998,15.4399,12.6583,13.8412,0.0199931,20.8425
3,1,1999,15.3679,12.3587,13.9283,0.0199931,25.9536
4,2,1997,14.8474,12.3572,13.6883,-0.00911877,43.5369
5,2,1998,14.7601,11.5891,13.6968,-0.0191691,41.273
6,2,1999,14.7038,12.6857,13.8871,-0.0292194,42.8179
7,3,1998,13.9124,12.1451,12.0143,0.145003,17.4865
8,3,1999,13.66,9.89273,12.0052,0.184223,17.4949
9,4,1995,17.6699,14.5626,17.1751,0.0532712,71.6182


Now we proceed to step 1 of Levin-Petrin 

So instead of the non-parametric part as a function of capital and investment, we make it of capital and materials 
This allows us to not filter out datasets where invesment equals 0 

In [222]:
# display(first(dataset, 10))

# A0: generating the polynomial terms 
dataset.v_capital_square = dataset.v_capital .* dataset.v_capital
dataset.v_material_square = dataset.v_material .* dataset.v_material
dataset.int_capital_material = dataset.v_capital .* dataset.v_material

# A1: running the step 1 regression 
step1 = lm(@formula(v_production ~ v_labor + v_capital + v_capital_square + v_material + v_material_square + int_capital_material), dataset)

display(step1)
β_labor = coef(step1)[2]

# B0: calculating the necessary variables for step 2 
# calculating residualized production without labor 
dataset.production_residuals = dataset.v_production - β_labor * dataset.v_labor 

# calculating predicted phi 
dataset.predicted_phi = (coef(step1)[1] .+ dataset.v_capital .* coef(step1)[3] .+
                            dataset.v_capital_square .* coef(step1)[4] .+ dataset.v_material .* coef(step1)[5] .+ 
                            dataset.v_material_square .* coef(step1)[6] .+
                            dataset.int_capital_material .* coef(step1)[7])

# now we delete the old variables 
select!(dataset, Not([:v_material_square, :v_capital_square]))

# # B0: now we calculate the lagged phi needed for step 2: lag_phi and lag_capital 
sort!(dataset, [:firm_id, :year])
transform!(groupby(dataset, :firm_id), :v_capital => (x -> lag(x, 1)) => :lag_capital) #generate capital lag variable
transform!(groupby(dataset, :firm_id), :predicted_phi => (x -> lag(x, 1)) => :lag_phi) # generate phi lag variable
transform!(groupby(dataset, :firm_id), :v_material => (x -> lag(x, 1)) => :lag_material) # generate phi lag variable

dataset = dropmissing(dataset, [:lag_capital, :lag_phi, :lag_material])

display(first(dataset, 10))

StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

v_production ~ 1 + v_labor + v_capital + v_capital_square + v_material + v_material_square + int_capital_material

Coefficients:
────────────────────────────────────────────────────────────────────────────────────────
                            Coef.   Std. Error      t  Pr(>|t|)    Lower 95%   Upper 95%
────────────────────────────────────────────────────────────────────────────────────────
(Intercept)            7.06978     0.201375     35.11    <1e-99   6.67505      7.46451
v_labor                0.0246518   0.000535489  46.04    <1e-99   0.0236021    0.0257014
v_capital              0.426918    0.0340238    12.55    <1e-35   0.360225     0.49361
v_capital_square       0.00754432  0.0014266     5.29    <1e-06   0.00474793   0.0103407
v_material            -1.02544     0.355336     -2.89    0.0039  -1.72196

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,int_capital_material,production_residuals,predicted_phi,lag_capital,lag_phi,lag_material
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1998,15.4399,12.6583,13.8412,0.0199931,20.8425,0.276729,14.9261,14.4182,13.6469,14.2947,0.0199931
2,1,1999,15.3679,12.3587,13.9283,0.0199931,25.9536,0.278471,14.7281,14.4737,13.8412,14.4182,0.0199931
3,2,1998,14.7601,11.5891,13.6968,-0.0191691,41.273,-0.262556,13.7426,14.3384,13.6883,14.3299,-0.00911877
4,2,1999,14.7038,12.6857,13.8871,-0.0292194,42.8179,-0.405774,13.6483,14.462,13.6968,14.3384,-0.0191691
5,3,1999,13.66,9.89273,12.0052,0.184223,17.4949,2.21163,13.2288,13.207,12.0143,13.2291,0.145003
6,4,1996,17.6065,15.2652,17.2278,0.08283,78.801,1.42698,15.6639,16.6533,17.1751,16.6208,0.0532712
7,5,1999,16.0031,14.0184,15.313,0.10276,51.9997,1.57356,14.7212,15.3527,15.1472,15.2477,0.0829572
8,6,1999,11.0108,8.07091,8.89698,0.0272698,-4.84692,0.242619,11.1303,11.45,8.54161,11.251,0.0272698
9,7,1997,12.5013,8.65869,9.51618,0.0,16.5183,0.0,12.0941,11.8156,9.18124,11.6254,0.0


# Step 2: running the GMM estimation to find $\beta_k$, $\beta_m$, and Markov parameters

The moment conditions being of [$\xi$ + $\epsilon$  | capital, material, lag_material, lag_capital]

Now we proceed to step 2 of the Levin-Petrin estimation procedure 

The difference between this compared to the OP procedure is that we create moment conditions of both X (technological shocks) and epsilon (error terms)

Because of that, to calculate the residual terms we deduct from v_production, so not just predicted_phi like I did in OP

And in the later GMM step we estimate beta_capital, the two alpha terms for Markov process, AND beta_material

In [223]:
# B0: defining the crucial functions for our GMM procedure 

function markov(parameter_guess, lag_phi, lag_capital, lag_material)
    beta_k = parameter_guess[1]
    beta_m = parameter_guess[2]
    alpha_0 = parameter_guess[3]
    alpha_1 = parameter_guess[4]

    # first-order markov process: now = alpha + alpha_1 * last
    value = alpha_0 .+ alpha_1 .* (lag_phi - beta_k .* lag_capital - beta_m .* lag_material)

    return value
end

function objective_function(
    parameter_guess, 
    residual_production, 
    lag_phi, 
    labor,
    capital, 
    lag_capital, 
    material,
    lag_material,
)
    beta_k = parameter_guess[1]
    beta_m = parameter_guess[2]
    alpha_0 = parameter_guess[3]
    alpha_1 = parameter_guess[4]    

    RHS = beta_k .* capital .+ beta_m .* material .+ markov(parameter_guess, lag_phi, lag_capital, lag_material)

    # this is the main difference compared to OP in my opinion 
    # residual production already netted out variations from labor 
    epsilon = residual_production .- RHS

    instrument = Matrix(hcat(ones(length(capital)), capital, lag_capital, material, lag_material))  # Nx5 matrix
    g = (transpose(epsilon) * instrument) ./ length(instrument)
    loss = dot(g, g)  # equivalent to g'T * g
    return loss
end 

objective_function (generic function with 1 method)

In [224]:
# tester function 
# display(first(dataset, 10))
objective_function([0,0,0,0], 
            dataset.v_production, dataset.lag_phi, 
            dataset.v_labor, dataset.v_capital, 
            dataset.lag_capital, dataset.v_material, 
            dataset.lag_material)

2288.2476082385974

In [225]:
# function to run the GMM procedure 
function GMM_main(
    initial_guess, 
    dataset
)
    result = optimize(parameter_guess -> objective_function(
            parameter_guess,
            dataset.production_residuals,
            dataset.lag_phi,
            dataset.v_labor,
            dataset.v_capital,
            dataset.lag_capital,
            dataset.v_material,
            dataset.lag_material,
        ),
        initial_guess,
        NelderMead()
    )
    min_loss = Optim.minimum(result)   # minimum loss value
    println("Minimum loss value: ", min_loss)

    parameter_estimated = Optim.minimizer(result)
    println(parameter_estimated)
    println("Estimated capital coefficient: ", parameter_estimated[1])
    println("Estimated material coefficient: ", parameter_estimated[2])
end 

GMM_main([0.1,0.1,0.1,0.1], dataset)

println("Estimated labor coefficient: ", β_labor)
println("Mean productivity values: ", mean(dataset.predicted_phi))

Minimum loss value: 1.2550893734429717e-6
[0.7700629449937111, -0.759376175074169, -0.16230379123153418, 1.0331792017414554]
Estimated capital coefficient: 0.7700629449937111
Estimated material coefficient: -0.759376175074169
Estimated labor coefficient: 0.024651755218352268
Mean productivity values: 13.299419480817049
